# Lab 7 Phase field simulation of dendritic solidification


#### <p style="text-align: right;"> &#9989; **put your name here** </p>

### Due 11/13/2025 
---
This assignment is mainly developed following R. Kobayashi's 1993 paper: *Ryo Kobayashi, Modeling and numerical simulations of dendritic crystal growth, Physica D 63 (1993) 410-423*.

At the end of this assignment, you should be able to create a model to simulating dendritic growth during solidification as in the video below.

In [ ]:
from IPython.display import YouTubeVideo

YouTubeVideo("VpP-nweuPCk",width=640,height=360)

## Part 1: Isotropic growth

The governing equations include one Allen-Cahn type phase field equation and one heat equation:

$$\frac{\partial \phi}{\partial t} = -L \bigg( \frac{\partial f}{\partial \phi} + E(T) - \bar{\varepsilon}^2 \nabla^2 \phi \bigg)$$

$$\frac{\partial T}{\partial t} = \kappa \nabla^2 T + h\frac{\partial \phi}{\partial t}.,$$

where Allen-Cahn eqaution is for order parameter that defines solid versus liquid, and heat equation for temperature. Note that we assume an isotropic surface energy here in the Allen-Cahn equation -- a constant gradient coefficient $\bar{\varepsilon}$. The  driving force for solidification is given as

$$E(T) = \frac{\alpha}{\pi} \phi \big(1-\phi \big) \tan^{-1} \big[ \zeta \big( T-T_m\big)\big].$$

We use a simple polynomial double-well free energy function 

$$f(\phi) = \frac{1}{4} \phi^2 \big(1-\phi \big)^2$$

with the two energy mimina at $\phi=0$ (lqiuid) and $\phi =1$ (solid), such that 

$$ \frac{\partial f}{\partial \phi} = \frac{1}{2}\phi \big(1-\phi \big) \big( 1- 2 \phi \big).$$


* Complete the code cells below to run a simulation of isotropic dendrtic growth. The material pamateres as given as $L = 1.0 / 0.0003$, $\bar{\varepsilon} = 0.01$, $\alpha = 0.9$, $\zeta = 10$, $h= 1.6$, and $T_m = 1.0$.

### Part 1.1. Parameter setup

In [ ]:
import numpy as np

# parameters

# 2D square domain
Ly = 9.0
Lx = 9.0

# discretization in the y and x directions
py = 300
px = 300

# grid spacing and time step size
dx = Lx / px
dy = Ly / py
dt = 0.0001
nstep = 4001

# constants
epsB = 0.01
L = 1.0 / 0.0003
kappa = 1.0
h = 1.6
alpha = 0.9
zeta = 10.
Tm = 1.0

# 2D arrays (initialized to zeros)
phi  = np.zeros((py, px))
mu   = np.zeros((py, px))
LapP = np.zeros((py, px))
E    = np.zeros((py, px))
T    = np.zeros((py, px))
LapT = np.zeros((py, px))


### Part 1.2. Initial condition

**Solidification is a type of nucleation and growth. Thus, we need to place initial nucleus in the computational box.**

* Place a seed that has a radius of grid spacings at the center of the domain. The distance to the center of the nucleus is

$$d = \sqrt{(x-x_c)^2 + (y-y_c)^2}.$$

For a nucleus of a radius of 5, we can set `if dis < 5`, $\phi$ `= 1`.

* Add some noise in the region between 8 grid spacings and the nucleus surface.

In [ ]:
# initial condition
radius = 5

for i in range(py):        # i = 0 ... py-1
    for j in range(px):    # j = 0 ... px-1
        # radius of the nucleus
        dis = np.sqrt( ?? )  
        if dis < radius:
            phi[i, j] = 1.0
        elif dis < ?? :
            phi[i, j] = np.random.rand()  # random number in [0,1)



# visualization
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time


fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # 1 row, 2 columns

# --- left plot ---
im0 = axes[0].imshow(phi, origin='upper', interpolation='nearest',
                     vmin=-0.01, vmax=1.01, cmap='viridis')
cbar0 = fig.colorbar(im0, ax=axes[0], label='phi')
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_title('phi')

# --- right plot ---
im1 = axes[1].imshow(T, origin='upper', interpolation='nearest',
                     vmin=0, vmax=1.2, cmap='hot')
cbar1 = fig.colorbar(im1, ax=axes[1], label='T')
axes[1].set_aspect('equal', adjustable='box')
axes[1].set_title('T')


### Part 1.3. Time evolution

We will use finite difference method for this simulation. The stencil is given as

$$\frac{\phi_{i,j}^{(n+1)}-\phi_{i,j}^{(n)}}{\Delta t} = - L \bigg(\frac{\partial f}{\partial \phi}\bigg|_{i,j}^{(n)} - \bar{\varepsilon}^2 \frac{\phi_{i,j-1}^{(n)} + \phi_{i,j+1}^{(n)} + \phi_{i-1,j}^{(n)} + \phi_{i+1,j}^{(n)} -4 \phi_{i,j}^{(n)}}{\Delta x^2} \bigg),$$

$$\frac{T_{i,j}^{(n+1)}-T_{i,j}^{(n)}}{\Delta t} = \kappa \frac{T_{i,j-1}^{(n)} + T_{i,j+1}^{(n)} + T_{i-1,j}^{(n)} + T_{i+1,j}^{(n)} -4 T_{i,j}^{(n)}}{\Delta x^2} + h \frac{\phi_{i,j}^{(n+1)}-\phi_{i,j}^{(n)}}{\Delta t} .$$

Complete the code below to run the simulation.

In [ ]:
tm = 0

for it in range(1, nstep + 1):

    # -------- order parameter field --------
    # driving force term E(T)
    E[1:py-1, 1:px-1] = ??

    # chemical potential mu (local double-well part)
    mu[1:py-1, 1:px-1] = ??

    # Laplacian of phi -> LapP
    # LapP[1:py-1, 1:px-1] = \
    #     (phi[0:py-2, 1:px-1] - 2.0 * phi[1:py-1, 1:px-1] + phi[2:py, 1:px-1]) / (dx**2) + \
    #     (phi[1:py-1, 0:px-2] - 2.0 * phi[1:py-1, 1:px-1] + phi[1:py-1, 2:px]) / (dy**2)
    
    # give yourself a test, use Python's slicing notation for vectorized calculation below.
    LapP[1:-1, 1:-1] = \
        (phi[??, 1:-1] - 2.0 * phi[1:-1, 1:-1] + phi[??, 1:-1]) / (dx**2) + \
        (phi[1:-1, ??] - 2.0 * phi[1:-1, 1:-1] + phi[1:-1, ??]) / (dy**2)

    
    # total chemical potential: mu = mu + E - epsB^2 * LapP
    mu[1:-1, 1:-1] = mu[1:-1, 1:-1]+ E[1:-1, 1:-1] - (epsB**2) * LapP[1:-1, 1:-1]

    
    # -------- temperature field --------
    # Laplacian of T -> LapT
    LapT[1:-1, 1:-1] = ??

    # -------- updates --------
    # update phi
    phi[1:py-1, 1:px-1] = phi[1:py-1, 1:px-1] - dt * L * mu[1:py-1, 1:px-1]

    # update T
    T[1:py-1, 1:px-1] = T[1:py-1, 1:px-1] + dt * LapT[1:py-1, 1:px-1] - \
        dt * h * L * mu[1:py-1, 1:px-1]

    # -------- no-flux boundary conditions (copy nearest interior) --------
    # top/bottom rows
    phi[0, :]    = phi[1, :]
    phi[py-1, :] = phi[py-2, :]
    # left/right cols
    phi[:, 0]    = phi[:, 1]
    phi[:, px-1] = phi[:, px-2]

    T[0, :]      = T[1, :]
    T[py-1, :]   = T[py-2, :]
    T[:, 0]      = T[:, 1]
    T[:, px-1]   = T[:, px-2]

    # advance time
    tm += dt

    # -------- visualization every 20 steps --------
    if it % 20 == 1:
        im0.set_data(phi)   
        im1.set_data(T)
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation 

&#9989; Do This - Describe the results you obtained. What do you see about the growth of solid region? What do you see about the temperature field?


---
## Part 2. Anisotropic growth.

Now, we will change the interfacial energy to be aniostopic. As we derived in the class, the Allen-Cahn becomes:

$$\frac{\partial \phi}{\partial t} = -L \bigg[ \frac{\partial f}{\partial \phi} + E(T) - \nabla \cdot \varepsilon(\theta)^2 \nabla^2 \phi + \frac{\partial}{\partial x}\varepsilon \varepsilon' \frac{\partial \phi}{\partial y} - \frac{\partial }{\partial y} \varepsilon \varepsilon' \frac{\partial \phi}{\partial x}\bigg].$$

This equation can be futher simplified to 

$$\frac{\partial \phi}{\partial t} = -L \bigg[ \frac{\partial f}{\partial \phi} + E(T) - \nabla \cdot \varepsilon(\theta)^2 \nabla^2 \phi + \frac{\partial \varepsilon \varepsilon'}{\partial x} \frac{\partial \phi}{\partial y} - \frac{\partial \varepsilon \varepsilon'}{\partial y}  \frac{\partial \phi}{\partial x}\bigg],$$

where the inward unit vector of the solid surface (solid-liquid interface) is 

$$\vec{n} = \frac{\nabla \phi}{\big| \nabla \phi \big|} = \big( n_x, n_y \big), $$

and the surface orientation is obtained as

$$\theta = \tan^{-1}\bigg(\frac{\phi,_y}{\phi,_x}\bigg).$$

The orientation-dependent interfacial energy is given by

$$\varepsilon(\theta) = \bar{\varepsilon} \big[ 1 + \delta \cos[J\cdot (\theta-\theta_0)] \big]$$

and its first derivative with respect to $\theta$ is

$$\varepsilon'(\theta) = - \bar{\varepsilon} J \delta \sin\big(J \cdot (\theta-\theta_0 ) \big).$$



Here, we define the interfacial region to be $\big| \nabla \phi \big| > 1.0\times10^{-6}$, and thus we calculate $\theta$ only for the region of $\big| \nabla \phi \big| > 1.0\times10^{-6}$.


For variable coefficient Laplacian, we can use the stencil:

$$\begin{split}\nabla \cdot \varepsilon(\theta)^2 \nabla \phi = \frac{1}{\Delta x} \bigg[ &\frac{\varepsilon(\theta)_{i,j+1}^2 + \varepsilon(\theta)_{i,j}^2}{2} \cdot \frac{\phi_{i,j+1}-\phi_{i,j}}{\Delta x} - \frac{\varepsilon(\theta)_{i,j}^2 + \varepsilon(\theta)_{i,j-1}^2}{2} \cdot\frac{\phi_{i,j}-\phi_{i,j-1}}{\Delta x} + \\
& \frac{\varepsilon(\theta)_{i+1,j}^2 + \varepsilon(\theta)_{i,j}^2}{2} \cdot\frac{\phi_{i+1,j}-\phi_{i,j}}{\Delta x} - \frac{\varepsilon(\theta)_{i,j}^2 + \varepsilon(\theta)_{i-1,j}^2}{2} \cdot\frac{\phi_{i,j}-\phi_{i-1,j}}{\Delta x} \bigg] \end{split}$$

or re-arranging to

$$ \begin{split}\nabla \cdot \varepsilon(\theta)^2 \nabla \phi = \frac{0.5}{\Delta x^2} \bigg[ & \big(\varepsilon(\theta)_{i,j+1}^2 + \varepsilon(\theta)_{i,j}^2 \big) \cdot \big(\phi_{i,j+1}-\phi_{i,j} \big) - \big( \varepsilon(\theta)_{i,j}^2 + \varepsilon(\theta)_{i,j-1}^2 \big) \cdot \big( \phi_{i,j}-\phi_{i,j-1}\big) + \\
& \big(\varepsilon(\theta)_{i+1,j}^2 + \varepsilon(\theta)_{i,j}^2 \big) \cdot\big(\phi_{i+1,j}-\phi_{i,j} \big) - \big(\varepsilon(\theta)_{i,j}^2 + \varepsilon(\theta)_{i-1,j}^2 \big) \cdot \big(\phi_{i,j}-\phi_{i-1,j} \big)  \bigg]. \end{split}$$


We can use central difference stencil for first order derivatives, for example,


$$\frac{\partial \varepsilon \varepsilon'}{\partial x} = \frac{\varepsilon \varepsilon'\big|_{i,j+1} - \varepsilon \varepsilon'\big|_{i,j-1}}{2\Delta x}$$

$$\frac{\partial \phi}{\partial y} = \frac{\phi_{i+1,j} -\phi_{i-1,j} }{2 \Delta y}.$$

As a result, 

$$\frac{\partial \varepsilon \varepsilon'}{\partial x} \frac{\partial \phi}{\partial y} = \frac{\varepsilon \varepsilon'\big|_{i,j+1} - \varepsilon \varepsilon'\big|_{i,j-1}}{2\Delta x} \cdot \frac{\phi_{i+1,j} -\phi_{i-1,j} }{2 \Delta y}.$$


The parameters for anisotropic surface energy is given as $\delta = 0.04$. 

* Implement the anisotropic interfacial energy to simulate dendritic growth.
* Set up initial condition and parameters.

In [ ]:
import numpy as np

# parameters

# 2D square domain
Ly = 9.0
Lx = 9.0

# discretization in the y and x directions
py = 300
px = 300

# grid spacing and time step size
dx = Lx / px
dy = Ly / py
dt = 0.0001
nstep = 4001

# constants
epsB = 0.01
L = 1.0 / 0.0003
kappa = 1.0
h = 1.6
alpha = 0.9
zeta = 10.
Tm = 1.0

# 2D arrays (initialized to zeros)
phi  = np.zeros((py, px))
mu   = np.zeros((py, px))
LapP = np.zeros((py, px))
E    = np.zeros((py, px))
T    = np.zeros((py, px))
LapT = np.zeros((py, px))


eps = np.zeros((py, px))
epsP = np.zeros((py, px))
theta = np.zeros((py, px))
grd = np.zeros((py, px, 2))
AvP = np.zeros((py, px))
epSQ = np.zeros((py, px))

term_dxdy = np.zeros((py, px))
term_dydx = np.zeros((py, px))

J = 6
delta = 0.04
theta0 = 0.5

# initial condition
for i in range(py):        # i = 0 ... py-1
    for j in range(px):    # j = 0 ... px-1
        rad = np.sqrt((??)  
        if rad < 5:
            phi[i, j] = 1.0
        elif rad < ??:
            phi[i, j] = np.random.rand()  # random number in [0,1)


# visualization
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time


fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # 1 row, 2 columns

# --- left plot ---
im0 = axes[0].imshow(phi, origin='upper', interpolation='nearest',
                     vmin=-0.01, vmax=1.01, cmap='viridis')
cbar0 = fig.colorbar(im0, ax=axes[0], label='phi')
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_title('phi')

# --- right plot ---
im1 = axes[1].imshow(T, origin='upper', interpolation='nearest',
                     vmin=0, vmax=1.2, cmap='hot')
cbar1 = fig.colorbar(im1, ax=axes[1], label='T')
axes[1].set_aspect('equal', adjustable='box')
axes[1].set_title('T')

* Complete the code below and run simulations.

In [ ]:

for it in range(1, nstep + 1):

    # -------- order parameter field --------
    # driving force term E(T)
    E[1:py-1, 1:px-1] = ??

    # chemical potential mu (local double-well part)
    mu[1:py-1, 1:px-1] = ??


    # calculate gradient of phi
    # dphi/dx
    grd[1:py-1, 1:px-1, 0] = ??
    # dphi/dy
    grd[1:py-1, 1:px-1, 1] = ?? 

    AvP[1:py-1, 1:px-1] = np.sqrt(grd[1:py-1, 1:px-1, 0]**2 + grd[1:py-1, 1:px-1, 1]**2)
    mask = AvP[1:py-1, 1:px-1] > 1e-6

    theta[1:py-1, 1:px-1][mask] = np.arctan2(grd[1:py-1, 1:px-1, 1][mask], \
        grd[1:py-1, 1:px-1, 0][mask])
    eps[1:py-1, 1:px-1][mask] = ??
    eps[1:py-1, 1:px-1][~mask] = 0
    epsP[1:py-1, 1:px-1][mask] = ??
    epsP[1:py-1, 1:px-1][~mask] = 0

    # add BC to eps and epsP
    eps[0, :]    = eps[1, :]
    eps[py-1, :] = eps[py-2, :]
    eps[:, 0]    = eps[:, 1]
    eps[:, px-1] = eps[:, px-2]  

    epsP[0, :]    = epsP[1, :]
    epsP[py-1, :] = epsP[py-2, :]
    epsP[:, 0]    = epsP[:, 1]
    epsP[:, px-1] = epsP[:, px-2]     

    epSQ[:, :] = eps[:, :]**2

    # variable coefficient Laplace (d epSQ d_phi/dx /dx)
    LapP[1:py-1, 1:px-1] = 0.5/dx**2 * \
        ((epSQ[1:py-1, 2:px  ]+epSQ[1:py-1, 1:px-1])*(phi[??, ??]-phi[??, ??]) - \
         (epSQ[1:py-1, 1:px-1]+epSQ[1:py-1, 0:px-2])*(phi[??, ??]-phi[??, ??]) + \
         (epSQ[2:py,   1:px-1]+epSQ[1:py-1, 1:px-1])*(phi[??, ??]-phi[??, ??]) - \
         (epSQ[1:py-1, 1:px-1]+epSQ[0:py-2, 1:px-1])*(phi[??, ??]-phi??, ??]))

    # (dee'/dx)*(dphi/dy)
    term_dxdy[1:-1, 1:-1] = \
        (eps[??,??]*epsP[??,??]-eps[??,??]*epsP[??,??])/(2*dx)* \
        (phi[??,??] - phi[??, ??])/(??*dx)
    
    # (dee'/dy)*(dphi/dx)
    term_dydx[1:-1, 1:-1] = ??

    # total chemical potential: mu = mu + E - epsB^2 * LapP
    # mu[1:py-1, 1:px-1] = mu[1:py-1, 1:px-1]+ E[1:py-1, 1:px-1] - \
    #     LapP[1:py-1, 1:px-1] + term_dxdy[1:py-1, 1:px-1] - term_dydx[1:py-1, 1:px-1]
    
    mu[1:-1, 1:-1] = mu[1:-1, 1:-1]+ E[1:-1, 1:-1] - \
        LapP[1:-1, 1:-1] + term_dxdy[1:-1, 1:-1] - term_dydx[1:-1, 1:-1]
    
    
    # -------- temperature field --------
    # Laplacian of T -> LapT
    LapT[1:py-1, 1:px-1] = ??

    # -------- updates --------
    # update phi
    phi[1:py-1, 1:px-1] = phi[1:py-1, 1:px-1] - dt * L * mu[1:py-1, 1:px-1]

    # update T
    T[1:py-1, 1:px-1] = T[1:py-1, 1:px-1] + dt * LapT[1:py-1, 1:px-1] - \
        dt * h * L * mu[1:py-1, 1:px-1]

    # -------- no-flux boundary conditions (copy nearest interior) --------
    # top/bottom rows
    phi[0, :]    = phi[1, :]
    phi[py-1, :] = phi[py-2, :]
    # left/right cols
    phi[:, 0]    = phi[:, 1]
    phi[:, px-1] = phi[:, px-2]

    T[0, :]      = T[1, :]
    T[py-1, :]   = T[py-2, :]
    T[:, 0]      = T[:, 1]
    T[:, px-1]   = T[:, px-2]

    # advance time
    tm += dt

    # -------- visualization every 20 steps --------
    if it % 20 == 1:
        im0.set_data(phi)   
        im1.set_data(T)
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation 

* Change the value of $J$ to 6 and re-run the simulations.

Do This - Describe the results you obtained. Descibe the difference between the two cases fo $J$ equals 4 and 6.

---
### Great! You're done. Please upload your completed file to the drop box on the course webpage.